# 🚀 Google Colab LLM Server (Laptop 1)
This notebook turns your Google Colab instance into a high-speed OpenAI-compatible API server with GPU acceleration and native **Tool / Function Calling** support.

### Step 1: Set Runtime
Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [ ]:
# [1] Install Ollama and Cloudflare Tunnel
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# [2] Start Ollama server & Download the Model
import os, subprocess, time

# Allow remote connections and bypass host checks
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"

print("Starting Ollama daemon...")
subprocess.Popen(["ollama", "serve"])
time.sleep(4)

# Qwen 2.5 7B offers top-tier instruction following and function calling
MODEL = "qwen2.5:7b"
print(f"Pulling model {MODEL}...")
!ollama pull $MODEL

In [ ]:
# [3] Launch Cloudflare Tunnel and Print Public Endpoint
import subprocess, re

# Pass --http-host-header localhost:11434 so Ollama accepts the proxied traffic
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--http-host-header", "localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
for line in tunnel_proc.stderr:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("=" * 60)
    print("🎉 YOUR COLAB LLM IS LIVE AND ACCESSIBLE GLOBALLY!")
    print(f"👉 COPY THIS URL TO LAPTOP 2:")
    print(f"   {public_url}")
    print("=" * 60)
else:
    print("Could not extract URL automatically. Check Colab output.")

tunnel_proc.wait()